# Import the required functions

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load Dataset

In [ ]:
df = pd.read_csv('C:\Datasets\insurance_2.csv')

# Understand the Data

In [ ]:
#Top 5 records
df.head()

In [ ]:
#Last 5 records
df.tail()

In [ ]:
#Get the number of rows and columns
df.shape

In [ ]:
#Columns data
df.columns

In [ ]:
#Check for null values. No null values
df.isnull().sum()

In [ ]:
#Check for duplicates and remove them if found
df[df.duplicated(keep=False)]

In [ ]:
# Remove duplicates from existing DataFrame
df.drop_duplicates(inplace=True)

In [ ]:
#Validate if duplicates are removed
df.duplicated().sum()

# Observation
- There are no null values in all the columns
- Initially there were 1338 records 
- There was one duplicate row which was removed

# Check the data type contained in the dataset

In [ ]:
# Check data types of all columns
print(df.dtypes)

In [ ]:
# Check the unique values of the text type fields
#sex
print(df['sex'].unique())  
#smoker
print(df['smoker'].unique())
#region
print(df['region'].unique())  

# Data Types in the dataset
- The dataset contains:
    - 3 Text types fields, i.e. sex, smoker and region
    - 4 Numeric fields, i.e. age, bmi, children and charges
    - Validated that uniqueness of the text type fields. I determined that there are no invalid entry in these fields

# Conduct Analysis
- Convert the text type fields to numeric
- Use **Backward Elimination** based on P-values. This method starts with all candidate predictors and removes the least significant variable (highest P-value) one by one until all remaining variables are below a significance threshold of 0.05

In [ ]:
#import statsmodels.api as sm

# Preprocessing: Convert categorical variables to dummy variables (numeric)
df_encoded = pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=True, dtype=int)
print(df_encoded)

# Define X and y
X = df_encoded.drop('charges', axis=1)
y = df_encoded['charges']

def backward_elimination(X, y, threshold=0.05):
    features = X.columns.tolist()
    while len(features) > 0:
        X_with_const = sm.add_constant(X[features])
        model = sm.OLS(y, X_with_const).fit()
        p_values = model.pvalues.drop('const')
        max_p_value = p_values.max()
        if max_p_value > threshold:
            excluded_feature = p_values.idxmax()
            features.remove(excluded_feature)
            print(f'Removed: {excluded_feature} with P-value {max_p_value:.4f}')
        else:
            break
    return features

selected_features = backward_elimination(X, y)
print(f'\nSelected Features: {selected_features}')

# Baseline Model Training 
- Train a standard **Linear Regression** model using the selected features. 
- Use the standard `LinearRegression` implementation from scikit-learn. Since LinearRegression uses **Ordinary Least Square**, no hyperparameters to be tuned
- The ratio is split like: 20% of your data is reserved for testing, while the remaining 80% is used for training

In [ ]:
X_baseline = X[selected_features]
X_train, X_test, y_train, y_test = train_test_split(X_baseline, y, test_size=0.2, random_state=42)

model_baseline = LinearRegression()
model_baseline.fit(X_train, y_train)

print("Model Trained.")

# Interpret and Evaluate Model
Evaluate the model using R-squared ($R^2$), Mean Absolute Error (MAE), and Root Mean Squared Error (RMSE).

In [ ]:
y_pred_baseline = model_baseline.predict(X_test)

r2_baseline = r2_score(y_test, y_pred_baseline)
mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))

print(f'Baseline R-squared: {r2_baseline:.4f}')
print(f'Baseline MAE: {mae_baseline:.4f}')
print(f'Baseline RMSE: {rmse_baseline:.4f}')

# Linear Regression Chart - Baseline Model
A scatter plot of Actual vs. Predicted values helps visualize the model's accuracy.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred_baseline)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Charges') 
plt.ylabel('Predicted Charges')
plt.title('Baseline Model: Actual vs Predicted Charges')
plt.show()

# Evaluate and Improve the Baseline Model

### Performance Metrics Justification
- **R-squared ($R^2$):** Indicates the proportion of variance for the dependent variable that's explained by the independent variables. Higher is better.
- **Mean Absolute Error (MAE):** Represents the average error magnitude in the same units as the target variable (dollars). It is less sensitive to outliers than MSE.
- **Root Mean Squared Error (RMSE):** Penalizes larger errors more heavily, which is useful in insurance where large cost underestimates can be risky.

# Retrain with Different Parameters
Medical insurance costs often show a synergistic effect between BMI and smoking. A smoker with a high BMI is much more likely to have extremely high medical charges than the sum of the two individual effects. Add an **interaction term** (`bmi * smoker`) to address this non-linearity.

In [ ]:
# Adding interaction between BMI and Smoking
print(df_encoded.columns)

df_refined = df_encoded.copy()

# Interaction terms
df_refined['bmi_smoker'] = df_refined['bmi'] * df_refined['smoker_yes']
df_refined['age_smoker'] = df_refined['age'] * df_refined['smoker_yes']

X_refined = df_refined[['age', 'bmi', 'smoker_yes', 'bmi_smoker', 'age_smoker']]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_refined, y, test_size=0.2, random_state=42)

model_refined = LinearRegression()
model_refined.fit(X_train_r, y_train_r)

print("Refined Model with Interaction Term Trained.")

In [ ]:
y_pred_refined = model_refined.predict(X_test_r)

r2_refined = r2_score(y_test_r, y_pred_refined)
mae_refined = mean_absolute_error(y_test_r, y_pred_refined)
rmse_refined = np.sqrt(mean_squared_error(y_test_r, y_pred_refined))

print(f'Refined R-squared: {r2_refined:.4f}')
print(f'Refined MAE: {mae_refined:.4f}')
print(f'Refined RMSE: {rmse_refined:.4f}')

print(f'\nImprovement in R2: {r2_refined - r2_baseline:.4f}')
print(f'Reduction in MAE: {mae_baseline - mae_refined:.4f}')

# Show Another Linear Regression Chart (Refined)
Visualizing the refined model reveals much better alignment along the diagonal, especially for higher charges.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test_r, y=y_pred_refined)
plt.plot([y_test_r.min(), y_test_r.max()], [y_test_r.min(), y_test_r.max()], 'g--', lw=2)
plt.xlabel('Actual Charges')
plt.ylabel('Predicted Charges')
plt.title('Refined Model: Actual vs Predicted Charges (with Interaction Term)')
plt.show()